In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
from src.preprocessing_utils import (
    load_raw_data,
    add_engineered_features,
    add_neighborhood_median,
    impute_missing_values,
    log_target,
)

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

In [3]:
df = load_raw_data("../data/raw/train.csv")
df = add_engineered_features(df)
df = add_neighborhood_median(df, df["SalePrice"])
df = impute_missing_values(df)

X = df.drop("SalePrice", axis=1)
y = log_target(df["SalePrice"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


cat_features = X.select_dtypes(include=["object"]).columns.tolist()

In [5]:
X.isnull().sum().sum()

np.int64(0)

In [6]:


model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    task_type="GPU",
    random_seed=42,
    verbose=False
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    verbose=False
)


CatBoostRegressor(depth=6, eval_metric='RMSE', iterations=3000, learning_rate=0.03, loss_function='RMSE', random_seed=42, task_type='GPU', verbose=False)

In [8]:
# get score on original scale
from sklearn.metrics import r2_score
import numpy as np


pred = model.predict(X_test)
r2_raw = r2_score(np.expm1(y_test), np.expm1(pred))
print("R2 on original scale:", r2_raw)


R2 on original scale: 0.9065609480911948
